In [19]:
!pip install lark posthog langchain langchain-core langchain-groq langchain_text_splitters langchain_huggingface
!pip install --upgrade langchain_huggingface
!pip install langchain_community
!pip install pypdf
!pip install sentence-transformers
!pip install langchain-classic

  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.3.5
    Uninstalling huggingface_hub-1.3.5:
      Successfully uninstalled huggingface_hub-1.3.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.0 which is incompatible.
  Using cached huggingface_hub-1.3.5-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-1.3.5-py3-none-any.whl (536 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into ac

In [22]:
def warn (*args , **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

In [23]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    groq_api_key = "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",
    max_tokens = 128,
    temperature = 0.5,
    
)

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import langchain
def text_splitter(data,chunk_size,chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len
    )
    chunks = text_splitter.split_documents(data)
    return chunks


In [25]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-V2"
)

Loading weights: 100%|█| 103/103 [00:00<00:00, 999.58it/s, Materializing param=poole
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-V2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
from langchain_community.document_loaders import TextLoader
import requests
file_name = "Company-Policies"
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/MZ9z1lm-Ui3YBp3SYWLTAQ/companypolicies.txt"
response = requests.get(url)
with open(file_name,'wb') as f1:
    print(f"Successfully written to file {file_name}")
    f1.write(response.content)
    

Successfully written to file Company-Policies


In [27]:
loader = TextLoader(file_name)
documents = loader.load()
print(f"The length of loaded documents is {len(documents)}")

The length of loaded documents is 1


In [7]:
documents

[Document(metadata={'source': 'Company-Policies'}, page_content="1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to con

In [8]:
chunk_txts = text_splitter(documents,200,20)

In [9]:
from langchain_community.vectorstores import Chroma

In [10]:
vectordb = Chroma.from_documents(chunk_txts, embeddings)

In [11]:
query = "Email policy"
retriever = vectordb.as_retriever(search_kwargs={"k":1})
docs = retriever.invoke(query)
docs[0].page_content

'3.\tInternet and Email Policy'

In [12]:
from langchain_community.document_loaders import PyPDFLoader

In [13]:
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf")
pdf_data = loader.load()
pdf_data[1]

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you. Its \ncore functionalities encompass: \n1. Context-Aware Capabilities: LangChain facilitates the \ndevelopment of applications that ar

In [14]:
chunks_pdf = text_splitter(pdf_data, 500 ,20)
ids = vectordb.get()['ids']
vectordb.delete(ids)
vectordb = Chroma.from_documents(chunks_pdf, embeddings)
print(langchain.__version__)

1.2.7


In [30]:
from langchain_classic.retrievers import MultiQueryRetriever
query = "What does the paper say about langchain?."
retriever = MultiQueryRetriever.from_llm(
    retriever = vectordb.as_retriever() ,llm = llm
)

In [34]:
docs = retriever.invoke(query)
docs


[Document(metadata={'page_label': '2', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'creator': 'Microsoft Word', 'total_pages': 6, 'page': 1, 'title': 's8329 final', 'creationdate': '2023-12-31T03:50:13+00:00', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'producer': 'PyPDF'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you. Its'),
 Document(metadata={'producer': 'PyPDF', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1ws